# Six organ-specific vasculature-to-FTU diagrams (Vega)

This notebook treats the six curated organ vascular CSVs as the authoritative
directed route records and `../data/Table_S1.csv` as the authoritative FTU
identity table. Organ applicability comes from each organ CSV; FTU status and
ontology identity must match Table_S1 and are never inferred from a visual
label, color, or hand-picked node.

The workflow is deliberately gated:

1. reconstruct every ordered source route from numeric `PathStep`;
2. compare the exact source nodes/edges with the exact model passed to Vega;
3. audit direction, reachability, route termination, duplicates, and
   source-row provenance;
4. construct Vega specs and write JSON/SVG files only if **all six organs PASS**.

Layout is presentation-only. It may change positions, spacing, label placement,
curve geometry, dimensions, and styling, but it must never change biological
nodes, FTU identity, ontology IDs, directed edges, branches, or route inclusion.

## 1. Paths, authoritative sources, and Vega helper

In [1]:
from pathlib import Path
import hashlib
import importlib
import json
import re
from collections import Counter, defaultdict, deque

import pandas as pd
import vl_convert as vlc

import organ_vasculature_vega as organ_vega

# Run All must use the helper currently saved beside this notebook, even when
# this kernel previously imported an older layout version.
organ_vega = importlib.reload(organ_vega)
ORGAN_ORDER = organ_vega.ORGAN_ORDER
build_organ_model = organ_vega.build_organ_model
make_vega_spec = organ_vega.make_vega_spec

PROJECT_DIR = Path(
    '/Users/zeynepamac/Desktop/Butterfly_v2.5_Update/'
    'ftus_connected_by_vasculature_per_organ'
)
CSV_DIR = PROJECT_DIR / 'organs_with_ftu_6_csv'
TABLE_S1_PATH = PROJECT_DIR.parent / 'data' / 'Table_S1.csv'
JSON_OUTPUT_DIR = PROJECT_DIR / 'organs_with_ftu_6_json'
SVG_OUTPUT_DIR = PROJECT_DIR / 'organs_with_ftu_6_svg'

assert CSV_DIR.is_dir(), CSV_DIR
assert TABLE_S1_PATH.is_file(), TABLE_S1_PATH
assert JSON_OUTPUT_DIR.is_dir(), JSON_OUTPUT_DIR
assert SVG_OUTPUT_DIR.is_dir(), SVG_OUTPUT_DIR
print('Vascular source folder:', CSV_DIR)
print('Authoritative FTU table:', TABLE_S1_PATH)
print('Vega JSON folder:', JSON_OUTPUT_DIR)
print('SVG folder:', SVG_OUTPUT_DIR)

Vascular source folder: /Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftus_connected_by_vasculature_per_organ/organs_with_ftu_6_csv
Authoritative FTU table: /Users/zeynepamac/Desktop/Butterfly_v2.5_Update/data/Table_S1.csv
Vega JSON folder: /Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftus_connected_by_vasculature_per_organ/organs_with_ftu_6_json
SVG folder: /Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftus_connected_by_vasculature_per_organ/organs_with_ftu_6_svg


## 2. Strict biological/topological validator definitions

No Vega specification is constructed in this section. Source direction is
derived from numeric `PathStep`, never from edge color. The independent source
reconstruction deliberately uses the same documented canonical identity rule
as the renderer—route role plus ontology ID, with a stable name key only where
the authoritative row omits an ontology ID—and then compares those independently
constructed sets with the model produced for Vega.

In [2]:
TABLE_COLUMNS = [
    "FTU",
    "FTUID",
    "Vessel",
    "VesselID",
    "PathStep",
    "PathVessel",
    "PathVesselID",
]
ROUTE_COLUMNS = ["FTUID", "FTU", "Vessel", "VesselID"]
HEART_NODE_ID = "heart:UBERON:0000948"


def _clean(value) -> str:
    return "" if pd.isna(value) else str(value).strip()


def _slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")


def _fallback_id(side: str, name: str) -> str:
    slug = _slug(name)
    digest = hashlib.sha1(slug.encode("utf-8")).hexdigest()[:10]
    return f"missing:{side}:{slug}:{digest}"


def _vessel_node_id(side: str, ontology_id: str, name: str) -> str:
    stable = _clean(ontology_id) or _fallback_id(side, _clean(name))
    return f"{side}:{stable}"


def _node_text(node_id: str, nodes: dict[str, dict]) -> str:
    node = nodes[node_id]
    ontology_id = node.get("ontology_id", "")
    suffix = f" [{ontology_id}]" if ontology_id else " [no ontology ID]"
    return f"{node['name']}{suffix}"


def _build_source_topology(frame: pd.DataFrame, organ_key: str) -> dict:
    frame = frame.copy().fillna("")
    frame["PathStep_num"] = pd.to_numeric(frame["PathStep"], errors="raise").astype(int)
    nodes = {
        HEART_NODE_ID: {
            "id": HEART_NODE_ID,
            "name": "Heart",
            "ontology_id": "UBERON:0000948",
            "type": "heart",
        }
    }
    edges: set[tuple[str, str, str]] = set()
    edge_routes: defaultdict[tuple[str, str, str], set[str]] = defaultdict(set)
    node_occurrences: Counter[str] = Counter()
    occurrence_records: defaultdict[str, list[dict]] = defaultdict(list)
    conflicts: list[str] = []
    routes: list[dict] = []

    def add_node(node: dict, route_id: str, path_step: int) -> None:
        existing = nodes.get(node["id"])
        signature = (node["name"], node["ontology_id"], node["type"])
        if existing is not None:
            old_signature = (
                existing["name"],
                existing["ontology_id"],
                existing["type"],
            )
            if signature != old_signature:
                conflicts.append(
                    f"{node['id']}: {old_signature!r} versus {signature!r}"
                )
        else:
            nodes[node["id"]] = node
        node_occurrences[node["id"]] += 1
        occurrence_records[node["id"]].append(
            {"route_id": route_id, "PathStep": path_step}
        )

    grouped = frame.groupby(ROUTE_COLUMNS, sort=True, dropna=False)
    for route_number, (route_key, group) in enumerate(grouped, 1):
        ftu_id, ftu_name, target_name, target_id = map(_clean, route_key)
        route_id = f"{organ_key}:route:{route_number:02d}"
        ordered = group.sort_values("PathStep_num", kind="stable")
        negative = ordered.loc[ordered["PathStep_num"] < 0]
        zero = ordered.loc[ordered["PathStep_num"] == 0]
        positive = ordered.loc[ordered["PathStep_num"] > 0]
        route_errors = []
        steps = ordered["PathStep_num"].tolist()
        if len(steps) != len(set(steps)):
            route_errors.append("duplicate PathStep values")
        if len(zero) != 1:
            route_errors.append(f"expected one PathStep 0 row; found {len(zero)}")
        if negative.empty:
            route_errors.append("missing arterial-origin side")
        if positive.empty:
            route_errors.append("missing venous-return side")
        if not negative.empty and negative["PathStep_num"].tolist() != list(
            range(int(negative["PathStep_num"].min()), 0)
        ):
            route_errors.append("non-contiguous negative PathStep sequence")
        if not positive.empty and positive["PathStep_num"].tolist() != list(
            range(1, int(positive["PathStep_num"].max()) + 1)
        ):
            route_errors.append("non-contiguous positive PathStep sequence")
        wrong_negative_types = negative.loc[negative["Type"].str.lower() != "artery"]
        wrong_positive_types = positive.loc[positive["Type"].str.lower() != "vein"]
        if len(wrong_negative_types):
            route_errors.append("negative PathStep row not labeled Artery")
        if len(wrong_positive_types):
            route_errors.append("positive PathStep row not labeled Vein")
        if len(zero) == 1:
            zero_row = zero.iloc[0]
            if _clean(zero_row["Type"]).lower() != "ftu":
                route_errors.append("PathStep 0 row not labeled FTU")
            if _clean(zero_row["PathVessel"]) != target_name:
                route_errors.append("PathStep 0 target name mismatch")
            if target_id and _clean(zero_row["PathVesselID"]) != target_id:
                route_errors.append("PathStep 0 target ID mismatch")

        artery_ids = []
        vein_ids = []
        for row in negative.to_dict("records"):
            node_id = _vessel_node_id(
                "artery", row["PathVesselID"], row["PathVessel"]
            )
            add_node(
                {
                    "id": node_id,
                    "name": _clean(row["PathVessel"]),
                    "ontology_id": _clean(row["PathVesselID"]),
                    "type": "artery",
                },
                route_id,
                int(row["PathStep_num"]),
            )
            artery_ids.append(node_id)
        for row in positive.to_dict("records"):
            node_id = _vessel_node_id(
                "vein", row["PathVesselID"], row["PathVessel"]
            )
            add_node(
                {
                    "id": node_id,
                    "name": _clean(row["PathVessel"]),
                    "ontology_id": _clean(row["PathVesselID"]),
                    "type": "vein",
                },
                route_id,
                int(row["PathStep_num"]),
            )
            vein_ids.append(node_id)

        ftu_node_id = f"ftu:{ftu_id}"
        add_node(
            {
                "id": ftu_node_id,
                "name": ftu_name,
                "ontology_id": ftu_id,
                "type": "ftu",
            },
            route_id,
            0,
        )

        artery_path = [HEART_NODE_ID, *artery_ids, ftu_node_id]
        vein_path = [ftu_node_id, *vein_ids, HEART_NODE_ID]
        route_edges = []
        for side, path in (("artery", artery_path), ("vein", vein_path)):
            for source_id, target_node_id in zip(path, path[1:]):
                edge = (source_id, target_node_id, side)
                edges.add(edge)
                edge_routes[edge].add(route_id)
                route_edges.append(edge)

        routes.append(
            {
                "route_id": route_id,
                "ftu_id": ftu_id,
                "ftu_name": ftu_name,
                "target_name": target_name,
                "target_id": target_id,
                "artery_path": artery_path,
                "vein_path": vein_path,
                "edges": route_edges,
                "errors": route_errors,
            }
        )

    return {
        "nodes": nodes,
        "edges": edges,
        "edge_routes": edge_routes,
        "node_occurrences": node_occurrences,
        "occurrence_records": occurrence_records,
        "node_conflicts": conflicts,
        "routes": routes,
    }


def _reachable(edges: set[tuple[str, str, str]], start: str, end: str, side: str) -> bool:
    adjacency: defaultdict[str, set[str]] = defaultdict(set)
    for source_id, target_id, edge_side in edges:
        if edge_side == side:
            adjacency[source_id].add(target_id)
    queue = deque([start])
    seen = {start}
    while queue:
        current = queue.popleft()
        if current == end:
            return True
        for neighbor in adjacency[current] - seen:
            seen.add(neighbor)
            queue.append(neighbor)
    return False


def validate_organ(
    organ_key: str,
    model: dict,
    table_s1: pd.DataFrame,
    table_row_set: set[tuple[str, ...]],
) -> dict:
    frame = pd.read_csv(model["source_path"], dtype=str).fillna("")
    source = _build_source_topology(frame, organ_key)
    authoritative_definitions = table_s1[["FTU", "FTUID"]].drop_duplicates()
    authoritative_by_name = dict(
        authoritative_definitions[["FTU", "FTUID"]].to_numpy()
    )
    organ_pairs = frame[["FTU", "FTUID"]].drop_duplicates()
    expected = {
        (row.FTUID, row.FTU)
        for row in organ_pairs.itertuples(index=False)
        if authoritative_by_name.get(row.FTU) == row.FTUID
    }
    authority_mismatches = [
        {
            "name": row.FTU,
            "source_id": row.FTUID,
            "Table_S1_id": authoritative_by_name.get(row.FTU),
        }
        for row in organ_pairs.itertuples(index=False)
        if authoritative_by_name.get(row.FTU) != row.FTUID
    ]
    found = {
        (node["ontology_id"], node["name"])
        for node in source["nodes"].values()
        if node["type"] == "ftu"
    }
    displayed = {
        (node["ontology_id"], node["name"])
        for node in model["nodes"]
        if node["type"] == "ftu"
    }
    missing_ftus = expected - found
    unexpected_ftus = found - expected
    displayed_not_authoritative = displayed - expected

    source_nodes = set(source["nodes"])
    visualization_nodes = {node["id"] for node in model["nodes"]}
    source_edges = source["edges"]
    visualization_edges = {
        (edge["source_id"], edge["target_id"], edge["side"])
        for edge in model["links"]
    }
    missing_nodes = source_nodes - visualization_nodes
    extra_nodes = visualization_nodes - source_nodes
    missing_edges = source_edges - visualization_edges
    extra_edges = visualization_edges - source_edges
    reversed_edges = {
        edge
        for edge in extra_edges
        if (edge[1], edge[0], edge[2]) in source_edges
    }
    disconnected_edges = {
        edge
        for edge in visualization_edges
        if edge[0] not in visualization_nodes or edge[1] not in visualization_nodes
    }
    impossible_edges = extra_edges - reversed_edges

    unreachable_ftus = []
    for ftu_id, ftu_name in sorted(expected):
        ftu_node_id = f"ftu:{ftu_id}"
        artery_ok = _reachable(
            visualization_edges, HEART_NODE_ID, ftu_node_id, "artery"
        )
        vein_ok = _reachable(
            visualization_edges, ftu_node_id, HEART_NODE_ID, "vein"
        )
        if not artery_ok or not vein_ok:
            unreachable_ftus.append(
                {
                    "id": ftu_id,
                    "name": ftu_name,
                    "arterial_reachable": artery_ok,
                    "venous_return_reachable": vein_ok,
                }
            )

    incorrect_routes = []
    for route in source["routes"]:
        missing_route_edges = set(route["edges"]) - visualization_edges
        if route["errors"] or missing_route_edges:
            incorrect_routes.append(
                {
                    "route_id": route["route_id"],
                    "errors": route["errors"],
                    "missing_edges": sorted(missing_route_edges),
                }
            )

    exact_duplicate_rows = frame.loc[frame.duplicated(keep=False)].copy()
    step_duplicate_rows = frame.loc[
        frame.duplicated(ROUTE_COLUMNS + ["PathStep"], keep=False)
    ].copy()
    ftu_route_occurrences = Counter(route["ftu_id"] for route in source["routes"])
    legitimate_repeated_ftus = {
        ftu_id: count for ftu_id, count in ftu_route_occurrences.items() if count > 1
    }
    graph_ftu_counts = Counter(
        node["ontology_id"]
        for node in model["nodes"]
        if node["type"] == "ftu"
    )
    duplicate_graph_ftu_ids = {
        ftu_id: count for ftu_id, count in graph_ftu_counts.items() if count > 1
    }

    ontology_to_nodes: defaultdict[str, list[str]] = defaultdict(list)
    for node_id, node in source["nodes"].items():
        ontology_id = node.get("ontology_id", "")
        if ontology_id:
            ontology_to_nodes[ontology_id].append(node_id)
    shared_ontology_nodes = {
        ontology_id: sorted(node_ids)
        for ontology_id, node_ids in ontology_to_nodes.items()
        if len(node_ids) > 1
    }
    repeated_node_occurrences = {
        node_id: count
        for node_id, count in source["node_occurrences"].items()
        if count > 1
    }

    source_rows_missing_from_table = [
        tuple(_clean(value) for value in row)
        for row in frame[TABLE_COLUMNS].to_numpy()
        if tuple(_clean(value) for value in row) not in table_row_set
    ]
    source_node_signatures = {
        node_id: (node["name"], node["ontology_id"], node["type"])
        for node_id, node in source["nodes"].items()
    }
    visualization_node_signatures = {
        node["id"]: (node["name"], node["ontology_id"], node["type"])
        for node in model["nodes"]
    }
    node_signature_mismatches = {
        node_id: {
            "source": source_node_signatures[node_id],
            "visualization": visualization_node_signatures[node_id],
        }
        for node_id in source_nodes & visualization_nodes
        if source_node_signatures[node_id] != visualization_node_signatures[node_id]
    }

    edge_route_mismatches = {}
    model_edge_by_key = {
        (edge["source_id"], edge["target_id"], edge["side"]): edge
        for edge in model["links"]
    }
    for edge in source_edges & visualization_edges:
        expected_routes = source["edge_routes"][edge]
        actual_routes = set(model_edge_by_key[edge]["routes"])
        if expected_routes != actual_routes:
            edge_route_mismatches[edge] = {
                "source_routes": sorted(expected_routes),
                "visualization_routes": sorted(actual_routes),
            }

    failures = []
    checks = {
        "source rows absent from Table_S1": source_rows_missing_from_table,
        "FTU authority mismatches": authority_mismatches,
        "missing FTUs": missing_ftus,
        "unexpected FTUs": unexpected_ftus,
        "displayed non-authoritative FTUs": displayed_not_authoritative,
        "duplicate graph FTU IDs": duplicate_graph_ftu_ids,
        "missing source nodes": missing_nodes,
        "extra visualization nodes": extra_nodes,
        "node signature mismatches": node_signature_mismatches,
        "missing source edges": missing_edges,
        "extra visualization edges": extra_edges,
        "edge route provenance mismatches": edge_route_mismatches,
        "reversed edges": reversed_edges,
        "disconnected edges": disconnected_edges,
        "impossible-direction edges": impossible_edges,
        "unreachable FTUs": unreachable_ftus,
        "incorrect route terminations": incorrect_routes,
        "source node identity conflicts": source["node_conflicts"],
        "exact duplicate dataframe rows": exact_duplicate_rows.to_dict("records"),
        "duplicate route/PathStep rows": step_duplicate_rows.to_dict("records"),
    }
    for label, values in checks.items():
        if len(values):
            failures.append(f"{label}: {len(values)}")

    return {
        "organ_key": organ_key,
        "display_name": model["display_name"],
        "frame": frame,
        "source": source,
        "expected_ftus": expected,
        "found_ftus": found,
        "displayed_ftus": displayed,
        "missing_ftus": missing_ftus,
        "unexpected_ftus": unexpected_ftus,
        "authority_mismatches": authority_mismatches,
        "legitimate_repeated_ftus": legitimate_repeated_ftus,
        "duplicate_graph_ftu_ids": duplicate_graph_ftu_ids,
        "missing_nodes": missing_nodes,
        "extra_nodes": extra_nodes,
        "missing_edges": missing_edges,
        "extra_edges": extra_edges,
        "reversed_edges": reversed_edges,
        "disconnected_edges": disconnected_edges,
        "impossible_edges": impossible_edges,
        "unreachable_ftus": unreachable_ftus,
        "incorrect_routes": incorrect_routes,
        "exact_duplicate_rows": exact_duplicate_rows,
        "step_duplicate_rows": step_duplicate_rows,
        "shared_ontology_nodes": shared_ontology_nodes,
        "repeated_node_occurrences": repeated_node_occurrences,
        "node_signature_mismatches": node_signature_mismatches,
        "edge_route_mismatches": edge_route_mismatches,
        "source_rows_missing_from_table": source_rows_missing_from_table,
        "failures": failures,
        "passed": not failures,
    }


def print_validation_report(report: dict, model: dict) -> None:
    source = report["source"]
    nodes = source["nodes"]
    print("\n" + "=" * 88)
    print(report["display_name"].upper())
    print("=" * 88)
    print("Authoritative FTUs:")
    for ftu_id, name in sorted(report["expected_ftus"]):
        print(f"  - {name} [{ftu_id}]")
    print("FTUs found in source graph:")
    for ftu_id, name in sorted(report["found_ftus"]):
        print(f"  - {name} [{ftu_id}]")
    print("Missing FTUs:", sorted(report["missing_ftus"]) or "none")
    print("Unexpected FTUs:", sorted(report["unexpected_ftus"]) or "none")
    print(
        "Legitimate repeated FTU route occurrences:",
        report["legitimate_repeated_ftus"] or "none",
    )
    print(
        "Duplicate graph FTU IDs:", report["duplicate_graph_ftu_ids"] or "none"
    )

    print("\nORDERED SOURCE ROUTES (direction comes from numeric PathStep)")
    for route in source["routes"]:
        target_id = f" [{route['target_id']}]" if route["target_id"] else " [no ID]"
        print(
            f"\n{route['route_id']} | {route['ftu_name']} [{route['ftu_id']}] "
            f"via {route['target_name']}{target_id}"
        )
        print(
            "  Arterial/source side: "
            + " -> ".join(_node_text(node_id, nodes) for node_id in route["artery_path"])
        )
        print(
            "  Venous/return side:   "
            + " -> ".join(_node_text(node_id, nodes) for node_id in route["vein_path"])
        )
        print("  Route errors:", route["errors"] or "none")

    print("\nSOURCE VS VEGA MODEL")
    print("  Missing source nodes:", sorted(report["missing_nodes"]) or "none")
    print("  Extra visualization nodes:", sorted(report["extra_nodes"]) or "none")
    print("  Missing source edges:", sorted(report["missing_edges"]) or "none")
    print("  Extra visualization edges:", sorted(report["extra_edges"]) or "none")
    print("  Reversed edges:", sorted(report["reversed_edges"]) or "none")
    print("  Disconnected edges:", sorted(report["disconnected_edges"]) or "none")
    print("  Impossible-direction edges:", sorted(report["impossible_edges"]) or "none")
    print("  Unreachable FTUs:", report["unreachable_ftus"] or "none")
    print("  Incorrect route terminations:", report["incorrect_routes"] or "none")

    print("\nGRAPH NODE DUPLICATION AUDIT")
    print(
        "  graph_int_id: NOT PROVIDED by the authoritative vascular CSV; "
        "canonical IDs below use route role + ontology ID (or stable missing-ID key)."
    )
    print(
        "  A. Exact duplicate dataframe rows:",
        len(report["exact_duplicate_rows"]),
    )
    if len(report["exact_duplicate_rows"]):
        duplicate_groups = report["exact_duplicate_rows"].groupby(
            list(report["exact_duplicate_rows"].columns),
            dropna=False,
            sort=True,
        ).size()
        for row_values, count in duplicate_groups.items():
            row = dict(zip(report["exact_duplicate_rows"].columns, row_values))
            print(
                "    - graph_int_id=NOT PROVIDED | "
                f"ontology_id={row.get('PathVesselID') or 'MISSING'} | "
                f"name={row.get('PathVessel')} | organ={report['organ_key']} "
                f"| occurrences={count}"
            )
    print(
        "  A. Duplicate route/PathStep rows:",
        len(report["step_duplicate_rows"]),
    )
    print("  B. Distinct role nodes sharing an ontology ID:")
    if report["shared_ontology_nodes"]:
        for ontology_id, node_ids in sorted(report["shared_ontology_nodes"].items()):
            for node_id in node_ids:
                node = source["nodes"][node_id]
                count = source["node_occurrences"].get(node_id, 0)
                print(
                    "    - graph_int_id=NOT PROVIDED | "
                    f"canonical_id={node_id} | ontology_id={ontology_id} | "
                    f"name={node['name']} | organ={report['organ_key']} "
                    f"| occurrences={count}"
                )
    else:
        print("    none")
    print(
        "  C. Legitimate distinct vascular instances: NOT DETERMINABLE because "
        "the source has no graph_int_id; no instance-level deduplication is performed."
    )
    print("  Repeated route occurrences of one canonical source entity:")
    if report["repeated_node_occurrences"]:
        for node_id, count in sorted(report["repeated_node_occurrences"].items()):
            node = source["nodes"][node_id]
            print(
                "    - graph_int_id=NOT PROVIDED | "
                f"canonical_id={node_id} | ontology_id={node['ontology_id'] or 'MISSING'} "
                f"| name={node['name']} | organ={report['organ_key']} "
                f"| occurrences={count}"
            )
    else:
        print("    none")

    visual_nodes = model["nodes"]
    print("\nEXACT INPUT TO VEGA (validated model, before make_vega_spec)")
    print("  Organ:", report["display_name"])
    print("  Root: Heart [UBERON:0000948]")
    print(
        "  FTUs:",
        [
            f"{node['name']} [{node['ontology_id']}]"
            for node in visual_nodes
            if node["type"] == "ftu"
        ],
    )
    print("  AS nodes: 0 (not part of this vascular-only Vega model)")
    print("  CT nodes: 0 (not part of this vascular-only Vega model)")
    print(
        "  arterial vessel nodes:",
        sum(node["type"] == "artery" for node in visual_nodes),
    )
    print(
        "  venous vessel nodes:",
        sum(node["type"] == "vein" for node in visual_nodes),
    )
    print("  total nodes:", len(visual_nodes))
    print("  total directed edges:", len(model["links"]))
    print(
        "  Every Vega node/edge has validated source provenance:",
        not any(
            (
                report["missing_nodes"],
                report["extra_nodes"],
                report["missing_edges"],
                report["extra_edges"],
                report["node_signature_mismatches"],
                report["edge_route_mismatches"],
            )
        ),
    )

    print("\nCONCISE PASS/FAIL")
    print(f"Expected FTUs: {len(report['expected_ftus'])}")
    print(f"Found FTUs: {len(report['found_ftus'])}")
    print(f"Missing FTUs: {len(report['missing_ftus'])}")
    print(f"Unexpected FTUs: {len(report['unexpected_ftus'])}")
    print(f"Duplicate FTU IDs: {len(report['duplicate_graph_ftu_ids'])}")
    print(f"Missing source nodes: {len(report['missing_nodes'])}")
    print(f"Extra visualization nodes: {len(report['extra_nodes'])}")
    print(f"Missing source edges: {len(report['missing_edges'])}")
    print(f"Extra visualization edges: {len(report['extra_edges'])}")
    direction_error_count = (
        len(report["reversed_edges"])
        + len(report["disconnected_edges"])
        + len(report["impossible_edges"])
        + len(report["incorrect_routes"])
    )
    print(f"Direction errors: {direction_error_count}")
    print(f"Unreachable FTUs: {len(report['unreachable_ftus'])}")
    print("# RESULT:", "PASS" if report["passed"] else "FAIL")
    if report["failures"]:
        print("# FAILURES:")
        for failure in report["failures"]:
            print("  -", failure)

## 3. Run full validation for all six organs

Table_S1 has no organ column. Therefore each curated organ CSV supplies organ
applicability, while Table_S1 independently authorizes each FTU name/ontology-ID
pair and every vascular route row. The full ordered paths and all discrepancy
sets are printed below.

In [3]:
table_s1 = pd.read_csv(TABLE_S1_PATH, dtype=str).fillna('')
missing_table_columns = sorted(set(TABLE_COLUMNS) - set(table_s1.columns))
if missing_table_columns:
    raise ValueError(f'Table_S1 missing required columns: {missing_table_columns}')

table_s1_definitions = table_s1[['FTU', 'FTUID']].drop_duplicates()
name_conflicts = table_s1_definitions.groupby('FTU')['FTUID'].nunique()
id_conflicts = table_s1_definitions.groupby('FTUID')['FTU'].nunique()
if (name_conflicts > 1).any() or (id_conflicts > 1).any():
    raise ValueError('Table_S1 FTU names and ontology IDs are not one-to-one')

table_row_set = {
    tuple(_clean(value) for value in row)
    for row in table_s1[TABLE_COLUMNS].to_numpy()
}
print(
    'Table_S1 authoritative inventory:',
    f'{table_s1_definitions.FTU.nunique()} FTU names /',
    f'{table_s1_definitions.FTUID.nunique()} ontology IDs /',
    f'{len(table_s1)} vascular rows',
)

models = {
    organ_key: build_organ_model(CSV_DIR, organ_key)
    for organ_key in ORGAN_ORDER
}
validation_reports = {}
for organ_key in ORGAN_ORDER:
    report = validate_organ(
        organ_key,
        models[organ_key],
        table_s1,
        table_row_set,
    )
    validation_reports[organ_key] = report
    print_validation_report(report, models[organ_key])

print('\n' + '=' * 88)
print('ALL-ORGAN VALIDATION GATE')
print('=' * 88)
for organ_key in ORGAN_ORDER:
    result = 'PASS' if validation_reports[organ_key]['passed'] else 'FAIL'
    print(f'{organ_key:16s}: {result}')

failed_organs = [
    organ_key
    for organ_key in ORGAN_ORDER
    if not validation_reports[organ_key]['passed']
]
ALL_VALIDATED = not failed_organs
print('ALL_VALIDATED =', ALL_VALIDATED)
if failed_organs:
    failure_details = {
        organ_key: validation_reports[organ_key]['failures']
        for organ_key in failed_organs
    }
    raise RuntimeError(
        'Vega rendering blocked because validation failed: '
        + json.dumps(failure_details, indent=2, default=str)
    )

validated_models = {
    organ_key: models[organ_key]
    for organ_key in ORGAN_ORDER
}

Table_S1 authoritative inventory: 28 FTU names / 28 ontology IDs / 1162 vascular rows

KIDNEY
Authoritative FTUs:
  - renal corpuscle [UBERON:0001229]
  - nephron [UBERON:0001285]
  - descending limb of loop of Henle [UBERON:0001289]
  - thick ascending limb of loop of Henle [UBERON:0001291]
  - loop of Henle ascending limb thin segment [UBERON:0004193]
  - cortical collecting duct [UBERON:0004203]
  - outer medullary collecting duct [UBERON:0004204]
  - inner medullary collecting duct [UBERON:0004205]
FTUs found in source graph:
  - renal corpuscle [UBERON:0001229]
  - nephron [UBERON:0001285]
  - descending limb of loop of Henle [UBERON:0001289]
  - thick ascending limb of loop of Henle [UBERON:0001291]
  - loop of Henle ascending limb thin segment [UBERON:0004193]
  - cortical collecting duct [UBERON:0004203]
  - outer medullary collecting duct [UBERON:0004204]
  - inner medullary collecting duct [UBERON:0004205]
Missing FTUs: none
Unexpected FTUs: none
Legitimate repeated FTU route

## 4. Vega input gate and rendering

The exact model counts are printed immediately before each Vega specification is
constructed. All six specifications are first validated in memory; files are
written only after every specification matches its validated model.

In [4]:
assert ALL_VALIDATED
validated_specs = {}

for organ_key in ORGAN_ORDER:
    model = validated_models[organ_key]
    report = validation_reports[organ_key]
    visual_nodes = model['nodes']
    visual_links = model['links']

    # Required exact pre-Vega input report.
    print('\n' + '-' * 88)
    print('Organ:', model['display_name'])
    print('Root: Heart [UBERON:0000948]')
    print(
        'FTUs:',
        [
            f"{node['name']} [{node['ontology_id']}]"
            for node in visual_nodes
            if node['type'] == 'ftu'
        ],
    )
    print('AS nodes: 0 (not part of this vascular-only Vega model)')
    print('CT nodes: 0 (not part of this vascular-only Vega model)')
    print(
        'arterial vessel nodes:',
        sum(node['type'] == 'artery' for node in visual_nodes),
    )
    print(
        'venous vessel nodes:',
        sum(node['type'] == 'vein' for node in visual_nodes),
    )
    print('total nodes:', len(visual_nodes))
    print('total directed edges:', len(visual_links))

    if not report['passed']:
        raise RuntimeError(f'{organ_key}: validation gate unexpectedly not PASS')
    if any((
        report['missing_nodes'], report['extra_nodes'],
        report['missing_edges'], report['extra_edges'],
        report['node_signature_mismatches'],
        report['edge_route_mismatches'],
    )):
        raise RuntimeError(f'{organ_key}: exact source/model comparison is not clean')

    spec = make_vega_spec(model)
    spec_data = {
        item['name']: item['values']
        for item in spec['data']
        if 'values' in item
    }
    expected_vessels = [
        node for node in visual_nodes if node['type'] in {'artery', 'vein'}
    ]
    expected_ftus = [node for node in visual_nodes if node['type'] == 'ftu']
    expected_heart = [node for node in visual_nodes if node['type'] == 'heart']
    assert spec_data['links'] == visual_links
    assert spec_data['vessels'] == expected_vessels
    assert spec_data['ftus'] == expected_ftus
    assert spec_data['heart'] == expected_heart
    assert spec['width'] == 3000
    validated_specs[organ_key] = spec
    print('Validated source/model correspondence: exact')

# No files are opened until all six in-memory specs have passed.
rendered = {}
for organ_key in ORGAN_ORDER:
    spec = validated_specs[organ_key]
    stem = f'{organ_key}_vasculature_ftus_vega'
    json_path = JSON_OUTPUT_DIR / f'{stem}.json'
    svg_path = SVG_OUTPUT_DIR / f'{stem}.svg'
    json_text = json.dumps(spec, indent=2, ensure_ascii=False) + '\n'
    svg_text = vlc.vega_to_svg(vg_spec=spec)
    if '<svg' not in svg_text or 'Heart' not in svg_text:
        raise ValueError(f'{organ_key}: incomplete Vega SVG render')
    json_path.write_text(json_text, encoding='utf-8')
    svg_path.write_text(svg_text, encoding='utf-8')
    rendered[organ_key] = {'json': json_path, 'svg': svg_path}
    print(f'{organ_key:16s} -> {json_path.name}; {svg_path.name}')


----------------------------------------------------------------------------------------
Organ: Kidney
Root: Heart [UBERON:0000948]
FTUs: ['cortical collecting duct [UBERON:0004203]', 'descending limb of loop of Henle [UBERON:0001289]', 'inner medullary collecting duct [UBERON:0004205]', 'loop of Henle ascending limb thin segment [UBERON:0004193]', 'nephron [UBERON:0001285]', 'outer medullary collecting duct [UBERON:0004204]', 'renal corpuscle [UBERON:0001229]', 'thick ascending limb of loop of Henle [UBERON:0001291]']
AS nodes: 0 (not part of this vascular-only Vega model)
CT nodes: 0 (not part of this vascular-only Vega model)
arterial vessel nodes: 17
venous vessel nodes: 14
total nodes: 40
total directed edges: 66
Validated source/model correspondence: exact

----------------------------------------------------------------------------------------
Organ: Liver
Root: Heart [UBERON:0000948]
FTUs: ['liver lobule [UBERON:0004647]']
AS nodes: 0 (not part of this vascular-only Vega model

## 5. Final regenerated-file verification and results

In [5]:
final_results = {}
for organ_key in ORGAN_ORDER:
    files = rendered[organ_key]
    spec = json.loads(files['json'].read_text(encoding='utf-8'))
    svg_text = files['svg'].read_text(encoding='utf-8')
    ftu_values = next(
        item['values'] for item in spec['data'] if item['name'] == 'ftus'
    )
    expected_ids = {
        ftu_id for ftu_id, _ in validation_reports[organ_key]['expected_ftus']
    }
    assert {node['ontology_id'] for node in ftu_values} == expected_ids
    assert spec['width'] == 3000
    svg_width = re.search(r'<svg[^>]*\bwidth="([^"]+)"', svg_text[:1000])
    assert svg_width and svg_width.group(1) == '3000'
    assert files['json'].is_file() and files['svg'].is_file()
    final_results[organ_key] = 'PASS'

print('\n' + '=' * 88)
print('FINAL VALIDATION AND REGENERATION RESULTS')
print('=' * 88)
for organ_key in ORGAN_ORDER:
    print(f'{organ_key:16s}: {final_results[organ_key]}')
print(
    'Complete: six validated Vega JSON files and six validated SVG files; '
    'no organ was rendered from a failing model.'
)


FINAL VALIDATION AND REGENERATION RESULTS
kidney          : PASS
liver           : PASS
large_intestine : PASS
small_intestine : PASS
mouth           : PASS
prostate        : PASS
Complete: six validated Vega JSON files and six validated SVG files; no organ was rendered from a failing model.
